In [ ]:
# ShadowTrace: Privacy leakage and Shannon Entropy audit
# Quantifying PII and secret leakage in raw OpenTelemetry traces

import os
import json
import math
import re
from collections import Counter
import pandas as pd

def calculate_shannon_entropy(text_string: str) -> float:
    # Calculating Shannon Entropy: H(X) = - sum(P(x_i) * log2(P(x_i)))
    if not text_string or not isinstance(text_string, str):
        return 0.0
    length = len(text_string)
    counts = Counter(text_string)
    return round(-sum((c / length) * math.log2(c / length) for c in counts.values()), 4)

REGEX_PATTERNS = {
    "Email": r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
    "JWT_Token": r'eyJh[a-zA-Z0-9_-]+\.eyJh[a-zA-Z0-9_-]+\.[a-zA-Z0-9_-]+',
    "Credit_Card": r'\b(?:\d[ -]*?){13,16}\b',
    "IPv4_Address": r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b'
}

def detect_regex_pii(text_string: str) -> list:
    if not isinstance(text_string, str):
        return []
    return [pii_type for pii_type, pattern in REGEX_PATTERNS.items() if re.search(pattern, text_string)]

def audit_trace_file(file_path: str, entropy_threshold: float = 3.5) -> pd.DataFrame:
    records = []
    if not os.path.exists(file_path):
        print(f"[!] Error: Trace file not found at {file_path}")
        return pd.DataFrame()

    with open(file_path, 'r') as f:
        data = json.load(f)

    batches = data.get("resourceSpans", []) if isinstance(data, dict) else []
    for batch in batches:
        service_name = "unknown_service"
        for attr in batch.get("resource", {}).get("attributes", []):
            if attr.get("key") == "service.name":
                service_name = attr.get("value", {}).get("stringValue", "unknown_service")

        for scope in batch.get("scopeSpans", []):
            for span in scope.get("spans", []):
                for attr in span.get("attributes", []):
                    key = attr.get("key", "")
                    val_obj = attr.get("value", {})
                    val_str = str(list(val_obj.values())[0]) if val_obj else ""

                    entropy = calculate_shannon_entropy(val_str)
                    pii_flags = detect_regex_pii(val_str)
                    is_high_entropy = entropy >= entropy_threshold

                    records.append({
                        "trace_id": span.get("traceId", ""),
                        "span_id": span.get("spanId", ""),
                        "service_name": service_name,
                        "span_name": span.get("name", ""),
                        "attribute_key": key,
                        "attribute_value_sample": val_str[:30] + "..." if len(val_str) > 30 else val_str,
                        "entropy": entropy,
                        "is_high_entropy": is_high_entropy,
                        "pii_detected": ", ".join(pii_flags) if pii_flags else "None",
                        "requires_redaction": is_high_entropy or len(pii_flags) > 0
                    })

    return pd.DataFrame(records)

if __name__ == "__main__":
    print("ShadowTrace privacy audit script started")